# Actividad 2: Modelos de Predicción y Clasificación en Machine Learning
* **Repositorio:** https://github.com/justinbeckham1204/Actividad2-Machine-Learning
* **Dataset:** `Wine Quality – Red Wine (Vinos tintos)` (Hugging Face Datasets)
* **Dataset URL:** : https://huggingface.co/datasets/codesignal/wine-quality

In [2]:

import pandas as pd
import numpy as np
import datetime
from datasets import load_dataset

# 0. Obtención automática del dataset desde Hugging Face
dataset_id = "codesignal/wine-quality"
print(f"Cargando dataset desde Hugging Face: {dataset_id}...")

hf_dataset = load_dataset(dataset_id, split="red")
df = pd.DataFrame(hf_dataset)

# Estandarización de nombres de columnas
df.columns = [c.strip().lower().replace(' ', '_').replace('-', '_') for c in df.columns]

# Convertir columnas numéricas
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Muestra del DataFrame e Inspección
print(f"Tamaño del DataFrame: {df.shape[0]} filas x {df.shape[1]} columnas")
print("\nTipos de datos actualizados:")
print(df.dtypes)
print("\nValores nulos reales detectados por columna:")
print(df.isnull().sum())

display(df.head())


Cargando dataset desde Hugging Face: codesignal/wine-quality...


Tamaño del DataFrame: 1599 filas x 12 columnas

Tipos de datos actualizados:
fixed_acidity           float64
volatile_acidity        float64
citric_acid             float64
residual_sugar          float64
chlorides               float64
free_sulfur_dioxide     float64
total_sulfur_dioxide    float64
density                 float64
ph                      float64
sulphates               float64
alcohol                 float64
quality                   int64
dtype: object

Valores nulos reales detectados por columna:
fixed_acidity           0
volatile_acidity        0
citric_acid             0
residual_sugar          0
chlorides               0
free_sulfur_dioxide     0
total_sulfur_dioxide    0
density                 0
ph                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [3]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Definición de columnas
num_features = [col for col in df.select_dtypes(include=[np.number]).columns if col != 'quality']
cat_features = df.select_dtypes(exclude=[np.number]).columns.tolist()

# Pipeline numérico con imputación por mediana y escalado
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocesador que combina variables numéricas y categóricas
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features)
])

print("Preprocesador configurado correctamente.")
print("\nVariables numéricas:", num_features)
print("Variables categóricas:", cat_features)


Preprocesador configurado correctamente.

Variables numéricas: ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
Variables categóricas: []


In [4]:

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Selección de variables predictoras y objetivo
X_reg = df.drop(columns=['quality'])
y_reg = df['quality']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Pipeline del modelo de regresión
reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Entrenamiento y predicción
reg_pipeline.fit(X_train_r, y_train_r)
y_pred_r = reg_pipeline.predict(X_test_r)

# Evaluación
mae_r = mean_absolute_error(y_test_r, y_pred_r)
rmse_r = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
r2_r = r2_score(y_test_r, y_pred_r)

print("--- RESULTADOS REGRESIÓN LINEAL ---")
print(f"MAE:  {mae_r:.4f}")
print(f"RMSE: {rmse_r:.4f}")
print(f"R²:   {r2_r:.4f}")


--- RESULTADOS REGRESIÓN LINEAL ---
MAE:  0.5035
RMSE: 0.6245
R²:   0.4032


In [5]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Creación de variable categórica a partir de la calidad del vino
def categorizar_calidad(quality):
    if quality <= 4:
        return 'Baja'
    elif quality <= 6:
        return 'Media'
    else:
        return 'Alta'

df['categoria_calidad'] = df['quality'].apply(categorizar_calidad)

# Exclusión de 'quality' y 'categoria_calidad' para evitar Target Leakage
X_cls = df.drop(columns=['quality', 'categoria_calidad'])
y_cls = df['categoria_calidad']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

# Pipeline de Clasificación con Random Forest
cls_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))
])

# Entrenamiento y predicción
cls_pipeline.fit(X_train_c, y_train_c)
y_pred_c = cls_pipeline.predict(X_test_c)

# Evaluación
acc_c = accuracy_score(y_test_c, y_pred_c)
prec_c = precision_score(y_test_c, y_pred_c, average='weighted', zero_division=0)
rec_c = recall_score(y_test_c, y_pred_c, average='weighted', zero_division=0)
f1_c = f1_score(y_test_c, y_pred_c, average='weighted', zero_division=0)

labels = ['Baja', 'Media', 'Alta']
cm_c = confusion_matrix(y_test_c, y_pred_c, labels=labels)

print("--- RESULTADOS CLASIFICACIÓN (Random Forest) ---")
print(f"Accuracy:  {acc_c:.4f}")
print(f"Precision: {prec_c:.4f}")
print(f"Recall:    {rec_c:.4f}")
print(f"F1-Score:  {f1_c:.4f}")

print("\nMatriz de Confusión:")
print(pd.DataFrame(cm_c, index=[f"Real {l}" for l in labels], columns=[f"Pred {l}" for l in labels]))

print("\nReporte de clasificación:")
print(classification_report(y_test_c, y_pred_c, labels=labels, zero_division=0))


--- RESULTADOS CLASIFICACIÓN (Random Forest) ---
Accuracy:  0.8469
Precision: 0.8001
Recall:    0.8469
F1-Score:  0.8075

Matriz de Confusión:
            Pred Baja  Pred Media  Pred Alta
Real Baja           0          13          0
Real Media          0         259          5
Real Alta           0          31         12

Reporte de clasificación:
              precision    recall  f1-score   support

        Baja       0.00      0.00      0.00        13
       Media       0.85      0.98      0.91       264
        Alta       0.71      0.28      0.40        43

    accuracy                           0.85       320
   macro avg       0.52      0.42      0.44       320
weighted avg       0.80      0.85      0.81       320



In [8]:

txt_filename = "resumen_modelos.txt"
now_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

txt_content = f"""======================================================================
RESUMEN AUTOMÁTICO DE EJECUCIÓN DE MODELOS DE MACHINE LEARNING
Fecha de ejecución: {now_str}
======================================================================

1. IDENTIFICACIÓN DEL DATASET Y PREPROCESAMIENTO
----------------------------------------------------------------------
- Dataset ID: codesignal/wine-quality (Hugging Face)
- Partición utilizada: red (vinos tintos)
- Enlace: https://huggingface.co/datasets/codesignal/wine-quality
- Registros totales: {df.shape[0]}
- Columnas actuales: {df.shape[1]}
- Variables predictoras: {len(num_features)}
- Variable objetivo para regresión: quality
- Preprocesamiento: imputación por mediana y escalado estándar.
- Prevención de Data Leakage: la variable objetivo se excluye de
  las características predictoras.

2. RESULTADOS DE REGRESIÓN LINEAL
----------------------------------------------------------------------
- Algoritmo: LinearRegression
- Variable objetivo: quality
- MAE: {mae_r:.4f}
- RMSE: {rmse_r:.4f}
- R²: {r2_r:.4f}

3. RESULTADOS DE CLASIFICACIÓN (RANDOM FOREST)
----------------------------------------------------------------------
- Algoritmo: RandomForestClassifier
- Variable objetivo: categoria_calidad
- Accuracy: {acc_c:.4f}
- Precision: {prec_c:.4f}
- Recall: {rec_c:.4f}
- F1-Score: {f1_c:.4f}

4. DISTRIBUCIÓN DE LAS CATEGORÍAS
----------------------------------------------------------------------
{df['categoria_calidad'].value_counts().to_string()}

5. MATRIZ DE CONFUSIÓN
----------------------------------------------------------------------
{pd.DataFrame(cm_c, index=[f"Real {l}" for l in labels], columns=[f"Pred {l}" for l in labels]).to_string()}

6. CONCLUSIONES
----------------------------------------------------------------------

- El modelo de Regresión Lineal obtuvo un R² de 0.4032, lo que
  indica que explica aproximadamente el 40.32% de la variabilidad
  de la calidad del vino. Su error absoluto medio fue de 0.5035.

- El modelo Random Forest alcanzó una exactitud del 84.69%.
  Sin embargo, presentó dificultades para identificar correctamente
  los vinos de calidad baja y alta.

- La distribución de las categorías está desbalanceada, ya que
  predominan los vinos de calidad media. Esto influye en el
  desempeño del modelo de clasificación.

- Los resultados muestran que ambos modelos permiten analizar
  la calidad del vino desde perspectivas diferentes: la regresión
  estima una puntuación numérica y la clasificación asigna una
  categoría de calidad.


======================================================================
FIN DEL INFORME
======================================================================
"""

with open(txt_filename, "w", encoding="utf-8") as file:
    file.write(txt_content)

print(f"Informe generado correctamente: {txt_filename}")



Informe generado correctamente: resumen_modelos.txt
